![IITIS](pictures/logoIITISduze.png)

# Algorytmu wyczerpującego przeszukiwania

Inna nazwa to brute force. Tutaj użyjemy potężniejszych wersji tego algorytmu. Ogólna ide jest 


## Implementacja naiwna

W naiwnej implementacji wyczerpującego przeszukiwania


In [2]:
import numpy as np
from math import inf
from itertools import product
from tqdm import tqdm

# Jak działa product

# Funkcja pomocnicza - liczenie energii
def calculate_energy(J: np.ndarray, h: np.ndarray, state: np.ndarray):
    return state @ J @ state.T + state @ h 



In [3]:
# Ćwiczenie - sprubuj samodzielnie zaimplementować wyszukiwanie naiwne

def brute_force_naive(J, h):
    best_energy = inf
    best_state = None

    # główna pętla
    ...
    
    return best_energy, best_state

In [4]:
# Przykładowa odpowiedź

def brute_force_naive(J, h):
    best_energy = inf
    best_state = None
    n = len(h)

    for state in tqdm(product([-1, 1], repeat=n), desc="Wyczerpujące przeszukiwanie", total=2**n):
        state = np.array(state)
        energy = calculate_energy(J, h, state)
        if energy < best_energy:
            best_energy = energy
            best_state = state

    return best_energy, best_state

In [5]:
# instancja testowa 16 spinów



In [6]:
# Polecam zobaczyć jak rozmiar n wpływa na czas obliczeń, dla n>20 czas obliczeń zaczyna rosnąć bardzo szybko
n = 16
rng = np.random.default_rng(seed=42)


scaling_func = np.vectorize(lambda x: 2*x-1)  # przesunięcie wartości z (0, 1) do (-1, 1)
J = np.triu(scaling_func(np.random.rand(n, n)))  # losowa gęsta macierz górnotrójkątna
h = scaling_func(np.random.rand(n))  # losowy wektor

sol_energy, sol_state = brute_force_naive(J, h)
print(sol_energy, sol_state)
print(J)
print(h)

Wyczerpujące przeszukiwanie:   0%|          | 0/65536 [00:00<?, ?it/s]

Wyczerpujące przeszukiwanie: 100%|██████████| 65536/65536 [00:00<00:00, 156063.40it/s]

-21.055200687863053 [-1 -1  1 -1 -1 -1 -1  1 -1 -1 -1 -1 -1  1  1 -1]
[[ 0.3129994  -0.02536343  0.71515108  0.17986078  0.9394086   0.36318741
   0.61571053 -0.24466755 -0.0867515  -0.7087096  -0.05775623 -0.19393758
  -0.97499461  0.15977085  0.11277722  0.60455166]
 [ 0.          0.68950045  0.36069518  0.67318696 -0.93638427  0.50955622
   0.40483645  0.77014482 -0.75338432  0.83727426 -0.15577407  0.32490868
  -0.02843584  0.08622053 -0.64773027 -0.83457283]
 [ 0.          0.          0.31445332  0.41653986 -0.54916619  0.8813124
  -0.17153332  0.98142986  0.3283994   0.2893651   0.3977915  -0.23189522
  -0.14436308 -0.77516795 -0.12607004  0.18867013]
 [ 0.          0.          0.         -0.56505918 -0.54224006  0.65137284
  -0.99140964  0.63411213 -0.00645457  0.99514122  0.6960945  -0.11102512
  -0.79181725  0.03235983  0.95674515 -0.966216  ]
 [ 0.          0.          0.          0.          0.60159275 -0.82810585
  -0.88310406  0.43978659  0.73660217 -0.57487711 -0.53916624

Jak widzać ta implementacja nie jest zbyt efektywna

## Podstawy obliczeń równoległych na CPU

In [7]:
from itertools import product
from threading import Thread, Lock
from tqdm import tqdm
import numpy as np
from math import inf


def brute_force_naive_multithreaded(J, h, num_threads=4):
    best_energy = inf
    best_state = None
    n = len(h)

    lock = Lock()

    def process_states(states_chunk):
        nonlocal best_energy, best_state
        for state in states_chunk:
            state = np.array(state)
            energy = calculate_energy(J, h, state)
            with lock:
                if energy < best_energy:
                    best_energy = energy
                    best_state = state


    all_states = list(product([-1, 1], repeat=n))

    chunk_size = len(all_states) // num_threads
    chunks = [all_states[i:i + chunk_size] for i in range(0, len(all_states), chunk_size)]
   
    if len(chunks) > num_threads:
        chunks[-2].extend(chunks[-1])
        chunks.pop()

    threads = []
    for chunk in chunks:
        thread = Thread(target=process_states, args=(chunk,))
        threads.append(thread)
        thread.start()


    for thread in threads:
        thread.join()

    return best_energy, best_state

In [10]:
n = 16

scaling_func = np.vectorize(lambda x: 2*x-1)  # przesunięcie wartości z (0, 1) do (-1, 1)
J = np.triu(scaling_func(np.random.rand(n, n)))  # losowa gęsta macierz górnotrójkątna
h = scaling_func(np.random.rand(n))  # losowy wektor


from dimod import BinaryQuadraticModel
from dwave.samplers import SimulatedAnnealingSampler

bqm_instance = BinaryQuadraticModel(h, J, vartype="SPIN")
sampler= SimulatedAnnealingSampler()
sampleset = sampler.sample(bqm_instance, num_reads=1)
print(sampleset)

s, e = brute_force_naive_multithreaded(J, h)
print(s)

   0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15    energy num_oc.
0 +1 +1 +1 -1 +1 +1 -1 -1 -1 +1 +1 -1 -1 -1 -1 +1 -29.71278       1
['SPIN', 1 rows, 1 samples, 16 variables]
-29.71278048321446
